In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"
SCHEMA = "ecom_poc"
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA ecom_poc")
# ==========================================
# 1. ORDERS: SAFE CASTING, VALIDATION & DEDUP
# ==========================================
bronze_orders = spark.table("bronze_orders")

parsed_orders = (
    bronze_orders
    .withColumn("clean_order_id", F.trim(F.col("order_id")))
    .withColumn("clean_customer_id", F.trim(F.col("customer_id")))
    .withColumn("parsed_order_date", F.expr("try_cast(trim(order_date) AS DATE)"))
    .withColumn("clean_category", F.initcap(F.trim(F.col("category"))))
    .withColumn("parsed_quantity", F.expr("try_cast(trim(quantity) AS INT)"))
    .withColumn("parsed_unit_price", F.expr("try_cast(trim(unit_price) AS DECIMAL(12,2))"))
    .withColumn("clean_order_status", F.upper(F.trim(F.col("order_status"))))
    .withColumn(
        "clean_country",
        F.when(F.upper(F.trim(F.col("country"))).isin("BD", "BANGLADESH"), F.lit("Bangladesh"))
         .otherwise(F.initcap(F.trim(F.col("country"))))
    )
)

validated_orders = parsed_orders.withColumn(
    "rejection_reasons",
    F.array_compact(F.array(
        F.when(F.col("clean_order_id").isNull() | (F.col("clean_order_id") == ""), F.lit("MISSING_ORDER_ID")),
        F.when(F.col("clean_customer_id").isNull() | (F.col("clean_customer_id") == ""), F.lit("MISSING_CUSTOMER_ID")),
        F.when(F.col("parsed_order_date").isNull(), F.lit("INVALID_ORDER_DATE")),
        F.when(F.col("parsed_order_date") > F.current_date(), F.lit("FUTURE_ORDER_DATE")),
        F.when(F.col("parsed_quantity").isNull() | (F.col("parsed_quantity") <= 0), F.lit("INVALID_QUANTITY")),
        F.when(F.col("parsed_unit_price").isNull() | (F.col("parsed_unit_price") <= 0), F.lit("INVALID_UNIT_PRICE")),
        F.when(~F.col("clean_order_status").isin("DELIVERED", "CANCELLED", "PENDING", "RETURNED"), F.lit("INVALID_ORDER_STATUS"))
    ))
)

# Outcome A: Quarantined Orders
quarantine_orders = validated_orders.filter(F.size("rejection_reasons") > 0)
quarantine_orders.write.mode("overwrite").saveAsTable("quarantine_orders")

# Valid Candidates -> Deterministic Deduplication (keep earliest _ingested_at, tie-break by _row_hash)
valid_orders = validated_orders.filter(F.size("rejection_reasons") == 0)
order_win = Window.partitionBy("clean_order_id").orderBy(F.col("_ingested_at").asc(), F.col("_row_hash").asc())
ranked_orders = valid_orders.withColumn("_rn", F.row_number().over(order_win))

# Outcome B: Accepted Silver Orders
silver_orders = (
    ranked_orders.filter(F.col("_rn") == 1)
    .select(
        F.col("clean_order_id").alias("order_id"),
        F.col("clean_customer_id").alias("customer_id"),
        F.col("parsed_order_date").alias("order_date"),
        F.col("clean_category").alias("category"),
        F.col("parsed_quantity").alias("quantity"),
        F.col("parsed_unit_price").alias("unit_price"),
        (F.col("parsed_quantity") * F.col("parsed_unit_price")).cast("DECIMAL(14,2)").alias("order_total_bdt"),
        F.col("clean_order_status").alias("order_status"),
        F.col("clean_country").alias("country"),
        "_ingested_at", "_source_file", "_batch_id", "_row_hash"
    )
)
silver_orders.write.mode("overwrite").saveAsTable("silver_orders")

# Outcome C: Duplicate Loser Orders
duplicate_orders = ranked_orders.filter(F.col("_rn") > 1)
duplicate_orders.write.mode("overwrite").saveAsTable("duplicate_loser_orders")


# ==========================================
# 2. TRANSACTIONS: SAFE CASTING, ORPHAN CHECK & DEDUP
# ==========================================
bronze_txns = spark.table("bronze_transactions")
valid_order_keys = spark.table("silver_orders").select("order_id").distinct()

parsed_txns = (
    bronze_txns
    .withColumn("clean_txn_id", F.trim(F.col("txn_id")))
    .withColumn("clean_order_id", F.trim(F.col("order_id")))
    .withColumn("clean_payment_method", F.trim(F.col("payment_method")))
    .withColumn("parsed_amount", F.expr("try_cast(trim(amount) AS DECIMAL(12,2))"))
    .withColumn("clean_currency", F.upper(F.trim(F.col("currency"))))
    .withColumn("clean_txn_status", F.upper(F.trim(F.col("txn_status"))))
    .withColumn("parsed_txn_ts", F.expr("try_cast(trim(txn_ts) AS TIMESTAMP)"))
)

txns_with_ref = parsed_txns.join(
    valid_order_keys.withColumn("_order_exists", F.lit(True)),
    parsed_txns["clean_order_id"] == valid_order_keys["order_id"],
    "left"
)

validated_txns = txns_with_ref.withColumn(
    "rejection_reasons",
    F.array_compact(F.array(
        F.when(F.col("clean_txn_id").isNull() | (F.col("clean_txn_id") == ""), F.lit("MISSING_TXN_ID")),
        F.when(F.col("_order_exists").isNull(), F.lit("ORPHAN_ORDER_ID")),
        F.when(F.col("parsed_amount").isNull() | (F.col("parsed_amount") <= 0), F.lit("INVALID_AMOUNT")),
        F.when(F.col("clean_currency") != "BDT", F.lit("UNSUPPORTED_CURRENCY")),
        F.when(~F.col("clean_txn_status").isin("SUCCESS", "FAILED", "PENDING", "REFUNDED"), F.lit("INVALID_TXN_STATUS")),
        F.when(F.col("parsed_txn_ts").isNull(), F.lit("INVALID_TIMESTAMP")),
        F.when(F.col("parsed_txn_ts") > F.current_timestamp(), F.lit("FUTURE_TIMESTAMP"))
    ))
)

# Outcome A: Quarantined Transactions
quarantine_txns = validated_txns.filter(F.size("rejection_reasons") > 0).drop("order_id", "_order_exists")
quarantine_txns.write.mode("overwrite").saveAsTable("quarantine_transactions")

valid_txns = validated_txns.filter(F.size("rejection_reasons") == 0)
txn_win = Window.partitionBy("clean_txn_id").orderBy(F.col("parsed_txn_ts").asc(), F.col("_row_hash").asc())
ranked_txns = valid_txns.withColumn("_rn", F.row_number().over(txn_win))

# Outcome B: Accepted Silver Transactions
silver_txns = (
    ranked_txns.filter(F.col("_rn") == 1)
    .select(
        F.col("clean_txn_id").alias("txn_id"),
        F.col("clean_order_id").alias("order_id"),
        F.col("clean_payment_method").alias("payment_method"),
        F.col("parsed_amount").alias("amount_bdt"),
        F.col("clean_currency").alias("currency"),
        F.col("clean_txn_status").alias("txn_status"),
        F.col("parsed_txn_ts").alias("txn_ts"),
        "_ingested_at", "_source_file", "_batch_id", "_row_hash"
    )
)
silver_txns.write.mode("overwrite").saveAsTable("silver_transactions")

# Outcome C: Duplicate Loser Transactions
duplicate_txns = ranked_txns.filter(F.col("_rn") > 1).drop("order_id", "_order_exists")
duplicate_txns.write.mode("overwrite").saveAsTable("duplicate_loser_transactions")


# ==========================================
# 3. EXACT RECONCILIATION INVARIANT CHECK
# ==========================================
for entity, b_tbl, s_tbl, q_tbl, d_tbl in [
    ("ORDERS", "bronze_orders", "silver_orders", "quarantine_orders", "duplicate_loser_orders"),
    ("TRANSACTIONS", "bronze_transactions", "silver_transactions", "quarantine_transactions", "duplicate_loser_transactions")
]:
    b_cnt = spark.table(b_tbl).count()
    s_cnt = spark.table(s_tbl).count()
    q_cnt = spark.table(q_tbl).count()
    d_cnt = spark.table(d_tbl).count()
    reconciled = (b_cnt == (s_cnt + q_cnt + d_cnt))
    
    print(f"=== {entity} RECONCILIATION ===")
    print(f"Bronze Total      : {b_cnt:,}")
    print(f"Accepted Silver   : {s_cnt:,}")
    print(f"Quarantined       : {q_cnt:,}")
    print(f"Duplicate Losers  : {d_cnt:,}")
    print(f"Sum of Outcomes   : {s_cnt + q_cnt + d_cnt:,} -> EXACT MATCH: {reconciled}\n")
    assert reconciled, f"Reconciliation failed for {entity}!"

=== ORDERS RECONCILIATION ===
Bronze Total      : 1,000,000
Accepted Silver   : 983,784
Quarantined       : 11,325
Duplicate Losers  : 4,891
Sum of Outcomes   : 1,000,000 -> EXACT MATCH: True

=== TRANSACTIONS RECONCILIATION ===
Bronze Total      : 1,000,000
Accepted Silver   : 966,372
Quarantined       : 28,884
Duplicate Losers  : 4,744
Sum of Outcomes   : 1,000,000 -> EXACT MATCH: True



In [0]:
%sql
USE workspace.ecom_poc;

-- 1. Quarantined row counts by rejection reason (Orders)
SELECT explode(rejection_reasons) AS rule_violated, COUNT(*) AS rejected_count
FROM quarantine_orders
GROUP BY 1 ORDER BY 2 DESC;

-- 2. Quarantined row counts by rejection reason (Transactions)
SELECT explode(rejection_reasons) AS rule_violated, COUNT(*) AS rejected_count
FROM quarantine_transactions
GROUP BY 1 ORDER BY 2 DESC;

-- 3. Sample Quarantined & Duplicate Loser rows
SELECT order_id, quantity, unit_price, order_date, rejection_reasons FROM quarantine_orders LIMIT 5;
SELECT clean_order_id, _rn, _ingested_at, _row_hash FROM duplicate_loser_orders LIMIT 5;

clean_order_id,_rn,_ingested_at,_row_hash
ORD-0000001,2,2026-11-18T08:50:19.424Z,4044af6be0efd866e2b7ca66a4a8f5510b5f7b1c2097dcd62c91592551a673ba
ORD-0000001,3,2026-11-18T08:50:19.424Z,c515f6bb8430035dcf8ac05d6dedc9eb9d1c83f2b10950cb2b3f0f837e1d2e71
ORD-0000002,2,2026-11-18T08:50:19.424Z,c63065a3051fb44dc68691965c712db26d348c46bf4203cd816f25d0c800e1fe
ORD-0000004,2,2026-11-18T08:50:19.424Z,416e9d3508f1eb0a90dcf2520f3ef4ed0caa2d74c03c0b0eff04fd8e483515bd
ORD-0000005,2,2026-11-18T08:50:19.424Z,e92e49aa6ff057730a9befa809bea345caec8d2aba8e99b1ea2f37fb254fb011
